In [1]:
import pandas as pd
xmas = pd.read_csv("https://www.dropbox.com/scl/fi/qxaslqqp5p08i1650rpc4/xmas.csv?rlkey=erdxi7jbh7pqf9fh4lv4cayp5&dl=1")
xmas

,Day,Day.in.Words,Gift.Item,Verb,Adjective,Location
0,1,first,partridge,NaN,NaN,in a pear tree
1,2,second,dove,NaN,turtle,NaN
2,3,third,hen,NaN,french,NaN
3,4,fourth,bird,NaN,calling,NaN
4,5,fifth,ring,NaN,golden,NaN
5,6,sixth,goose,a-laying,NaN,NaN
6,7,seventh,swan,a-swimming,NaN,NaN
7,8,eighth,maid,a-milking,NaN,NaN
8,9,ninth,lady,dancing,NaN,NaN
9,10,tenth,lord,a-leaping,NaN,NaN


Function 1: pluralize_gift()

In [2]:
def pluralize_gift(gift):
    """
    Return the plural form of a gift noun.

    Parameters
    ----------
    gift : str
        A singular noun.

    Returns
    -------
    str
        The plural form of the noun.
    """
    gift = str(gift)

    if "oo" in gift:
        gift = gift.replace("oo", "ee")
    elif gift.endswith("y"):
        gift = gift[:-1] + "ies"
    else:
        gift = gift + "s"

    return gift

In [3]:
assert pluralize_gift("goose") == "geese"
assert pluralize_gift("lady") == "ladies"
assert pluralize_gift("ring") == "rings"

xmas["Plural.Gift"] = xmas["Gift.Item"].apply(pluralize_gift)
xmas[["Gift.Item", "Plural.Gift"]]

,Gift.Item,Plural.Gift
0,partridge,partridges
1,dove,doves
2,hen,hens
3,bird,birds
4,ring,rings
5,goose,geese
6,swan,swans
7,maid,maids
8,lady,ladies
9,lord,lords


Function 2: make_phrase()

In [4]:
def make_phrase(num, num_word, item, verb, adjective, location):
    """
    Construct one gift phrase for a verse of the song.

    Parameters
    ----------
    num : int
        Day number.
    num_word : str
        Cardinal form of the number, such as 'ten'.
    item : str
        Singular gift noun.
    verb, adjective, location : str or missing
        Optional words that describe the gift.

    Returns
    -------
    str
        A complete gift phrase.
    """
    verb = "" if pd.isna(verb) else str(verb).strip()
    adjective = "" if pd.isna(adjective) else str(adjective).strip()
    location = "" if pd.isna(location) else str(location).strip()
    item = str(item).strip()

    if int(num) > 1:
        item = pluralize_gift(item)

    if int(num) == 1:
        article_or_number = "an" if item[0].lower() in "aeiou" else "a"
    else:
        article_or_number = str(num_word).strip()

    pieces = [article_or_number, adjective, item, verb, location]
    phrase = " ".join(piece for piece in pieces if piece)

    return phrase

In [5]:
number_words = {
    1: "one",
    2: "two",
    3: "three",
    4: "four",
    5: "five",
    6: "six",
    7: "seven",
    8: "eight",
    9: "nine",
    10: "ten",
    11: "eleven",
    12: "twelve",
}

xmas["Number.Word"] = xmas["Day"].map(number_words)
xmas[["Day", "Day.in.Words", "Number.Word"]]

,Day,Day.in.Words,Number.Word
0,1,first,one
1,2,second,two
2,3,third,three
3,4,fourth,four
4,5,fifth,five
5,6,sixth,six
6,7,seventh,seven
7,8,eighth,eight
8,9,ninth,nine
9,10,tenth,ten


In [6]:
assert make_phrase(
    num=10,
    num_word="ten",
    item="lord",
    verb="a-leaping",
    adjective="",
    location="",
) == "ten lords a-leaping"

assert make_phrase(1, "one", "email", "", "", "from Cal Poly") == (
    "an email from Cal Poly"
)
assert make_phrase(6, "six", "goose", "a-laying", "", "") == (
    "six geese a-laying"
)
assert make_phrase(9, "nine", "lady", "dancing", "", "") == (
    "nine ladies dancing"
)

xmas["Full.Phrase"] = xmas.apply(
    lambda row: make_phrase(
        row["Day"],
        row["Number.Word"],
        row["Gift.Item"],
        row["Verb"],
        row["Adjective"],
        row["Location"],
    ),
    axis=1,
)

xmas[["Day", "Full.Phrase"]]

,Day,Full.Phrase
0,1,a partridge in a pear tree
1,2,two turtle doves
2,3,three french hens
3,4,four calling birds
4,5,five golden rings
5,6,six geese a-laying
6,7,seven swans a-swimming
7,8,eight maids a-milking
8,9,nine ladies dancing
9,10,ten lords a-leaping


Function 3: sing_day()

In [7]:
def sing_day(dataset, num, phrase_col):
    """
    Return one complete verse of a cumulative gift song.

    Parameters
    ----------
    dataset : pandas.DataFrame
        Song data containing Day, Day.in.Words, and a phrase column.
    num : int
        The day to sing.
    phrase_col : str
        Name of the column containing complete gift phrases.

    Returns
    -------
    str
        One formatted verse.
    """
    num = int(num)

    if num not in dataset["Day"].values:
        raise ValueError("num must be a day contained in the dataset")
    if phrase_col not in dataset.columns:
        raise ValueError(f"{phrase_col!r} is not a column in the dataset")

    num_word = dataset.loc[
        dataset["Day"] == num, "Day.in.Words"
    ].iloc[0]
    intro = (
        f"On the {num_word} day of Christmas, "
        "my true love gave to me:"
    )

    phrases = (
        dataset.loc[dataset["Day"] <= num]
        .sort_values("Day", ascending=False)[phrase_col]
        .tolist()
    )

    gift_lines = []
    for index, phrase in enumerate(phrases):
        phrase = str(phrase).strip()

        if index == len(phrases) - 1:
            gift_lines.append(phrase + ".")
        elif index == len(phrases) - 2:
            gift_lines.append(phrase + ", and")
        else:
            gift_lines.append(phrase + ",")

    return intro + "\n" + "\n".join(gift_lines)

In [8]:
expected_third_day = (
    "On the third day of Christmas, my true love gave to me:\n"
    "three french hens,\n"
    "two turtle doves, and\n"
    "a partridge in a pear tree."
)

assert sing_day(xmas, 3, "Full.Phrase") == expected_third_day
print(sing_day(xmas, 3, "Full.Phrase"))

On the third day of Christmas, my true love gave to me:
three french hens,
two turtle doves, and
a partridge in a pear tree.


In [9]:
days = pd.Series(range(1, len(xmas) + 1))
verses = days.map(lambda day: sing_day(xmas, day, "Full.Phrase"))
full_xmas_song = "\n\n".join(verses)

print(full_xmas_song)

On the first day of Christmas, my true love gave to me:
a partridge in a pear tree.

On the second day of Christmas, my true love gave to me:
two turtle doves, and
a partridge in a pear tree.

On the third day of Christmas, my true love gave to me:
three french hens,
two turtle doves, and
a partridge in a pear tree.

On the fourth day of Christmas, my true love gave to me:
four calling birds,
three french hens,
two turtle doves, and
a partridge in a pear tree.

On the fifth day of Christmas, my true love gave to me:
five golden rings,
four calling birds,
three french hens,
two turtle doves, and
a partridge in a pear tree.

On the sixth day of Christmas, my true love gave to me:
six geese a-laying,
five golden rings,
four calling birds,
three french hens,
two turtle doves, and
a partridge in a pear tree.

On the seventh day of Christmas, my true love gave to me:
seven swans a-swimming,
six geese a-laying,
five golden rings,
four calling birds,
three french hens,
two turtle doves, and
a 

In [10]:
xmas2_url = (
    "https://www.dropbox.com/scl/fi/p9x9k8xwuzs9rhp582vfy/xmas_2.csv"
    "?rlkey=kvc3j3lmyn4opcidsrhcmrof1&dl=1"
)

xmas2 = pd.read_csv(xmas2_url)
xmas2["Number.Word"] = xmas2["Day"].map(number_words)

xmas2["Full.Phrase"] = xmas2.apply(
    lambda row: make_phrase(
        row["Day"],
        row["Number.Word"],
        row["Gift.Item"],
        row["Verb"],
        row["Adjective"],
        row["Location"],
    ),
    axis=1,
)

xmas2[["Day", "Full.Phrase"]]

,Day,Full.Phrase
0,1,an email from Cal Poly
1,2,two meal points
2,3,three lost pens
3,4,four course reviews
4,5,five practice exams
5,6,six graders grading
6,7,seven seniors stressing
7,8,eight moms a-calling
8,9,nine parties bumping
9,10,ten loads of laundry


In [ ]:
surprise_days = pd.Series(range(1, len(xmas2) + 1))
surprise_verses = surprise_days.map(
    lambda day: sing_day(xmas2, day, "Full.Phrase")
)
surprise_song = "\n\n".join(surprise_verses)

print(surprise_song)

On the first day of Christmas, my true love gave to me:
an email from Cal Poly.

On the second day of Christmas, my true love gave to me:
two meal points, and
an email from Cal Poly.

On the third day of Christmas, my true love gave to me:
three lost pens,
two meal points, and
an email from Cal Poly.

On the fourth day of Christmas, my true love gave to me:
four course reviews,
three lost pens,
two meal points, and
an email from Cal Poly.

On the fifth day of Christmas, my true love gave to me:
five practice exams,
four course reviews,
three lost pens,
two meal points, and
an email from Cal Poly.

On the sixth day of Christmas, my true love gave to me:
six graders grading,
five practice exams,
four course reviews,
three lost pens,
two meal points, and
an email from Cal Poly.

On the seventh day of Christmas, my true love gave to me:
seven seniors stressing,
six graders grading,
five practice exams,
four course reviews,
three lost pens,
two meal points, and
an email from Cal Poly.

On t